<a href="https://colab.research.google.com/github/francji1/01ZLMA/blob/main/assignment/01ZLMA_assignment_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment for Course 01ZLMA in 2024/2025

The assignment should be completed on patient data where heart disease was diagnosed.

The original dataset can be found here: https://archive.ics.uci.edu/ml/datasets/Heart+Disease

Various analyses and visualizations of this dataset can also be found here: https://www.kaggle.com/ronitf/heart-disease-uci (another notation, different NaN manipulation, etc ...)

However, for this assignment, the data have been slightly modified and split as available from the link below.

## 00 - Data description


    age:
    sex:
        0: Female
        1: Male
    chest_pain_type: Chest Pain Type
        0: asymptomatic
        1: atypical angina
        2: non-anginal pain
        3: typical angina
    blood_pressure: Resting Blood Pressure: Person's resting blood pressure
    cholesterol: Serum Cholesterol in mg/dl
    blood_sugar: Fasting Blood Sugar
        0:Less Than 120mg/ml
        1: Greater Than 120mg/ml
    rest_ecg: Resting Electrocardiographic Measurement
        0: showing probable or definite left ventricular hypertrophy by Estes' criteria
        1: normal
        2: having ST-T wave abnormality (T wave inversions and/or ST elevation or depression of > 0.05 mV)
    heart_rate: Max Heart Rate Achieved: Maximum Heart Rate Achieved
    ex_angina: Exercise Induced Angina
        1: Yes
        0: No
    st_depression: ST depression induced by exercise relative to rest
    st_slope: Slope of the peak exercise ST segment
        0: downsloping
        1: flat
        2: upsloping
    thal:  blood disorder called 'Thalassemia':
        1: fixed defect
        2: normal
        3: reversable Defect
    num_vessels: Number of Major Vessels: Number of major vessels colored by fluoroscopy


### Loading and preprocessing data


In [1]:
import io

import numpy as np
import pandas as pd
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats


In [2]:
BASE_URL = "https://raw.githubusercontent.com/francji1/01ZLMA/main/data/"

def load_csv(name, sep=","):
    r = requests.get(BASE_URL + name, verify=False)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), sep=sep)

data_train = load_csv("heart_train.csv")
data_train.head()


,age,sex,chest_pain_type,blood_pressure,cholesterol,blood_sugar,rest_ecg,heart_rate,ex_angina,st_depression,st_slope,num_vessels,thal,disease
0,63,male,3,145,233,1,0,150,0,2.3,0,0,1,0
1,37,male,2,130,250,0,1,187,0,3.5,0,0,2,0
2,41,female,1,130,204,0,0,172,0,1.4,2,0,2,0
3,56,male,1,120,236,0,1,178,0,0.8,2,0,2,0
4,57,female,0,120,354,0,1,163,1,0.6,2,0,2,0


In [3]:
data_test = load_csv("heart_test.csv")
data_test.head()


,age,sex,chest_pain_type,blood_pressure,cholesterol,blood_sugar,rest_ecg,heart_rate,ex_angina,st_depression,st_slope,num_vessels,thal,disease
0,66,1,0,160,228,0,0,138,0,2.3,2,0,1,0
1,71,0,0,112,149,0,1,125,0,1.6,1,0,2,0
2,64,1,3,170,227,0,0,155,0,0.6,1,0,3,0
3,66,0,2,146,278,0,0,152,0,0.0,1,1,2,0
4,39,0,2,138,220,0,1,152,0,0.0,1,0,2,0


### Creating an aggregated table

In [4]:
data_table = (
    data_train[["age", "sex", "blood_pressure", "disease"]]
    .assign(
        age=pd.cut(
            data_train["age"],
            bins=[-np.inf, 44, 60, np.inf],
            labels=["≤44", "45-60", ">60"],
        ),
        blood_pressure=pd.cut(
            data_train["blood_pressure"],
            bins=[-np.inf, 120, 130, 140, np.inf],
            labels=["≤120", "121-130", "131-140", ">140"],
        ),
    )
    .groupby(["age", "blood_pressure"], observed=True)
    .agg(n=("disease", "size"),
         disease_yes=("disease", "sum"))
    .assign(disease_no=lambda d: d["n"] - d["disease_yes"])
    .reset_index()
)
data_table


,age,blood_pressure,n,disease_yes,disease_no
0,≤44,≤120,26,7,19
1,≤44,121-130,11,1,10
2,≤44,131-140,9,3,6
3,≤44,>140,3,1,2
4,45-60,≤120,44,15,29
5,45-60,121-130,42,22,20
6,45-60,131-140,35,14,21
7,45-60,>140,33,21,12
8,>60,≤120,18,9,9
9,>60,121-130,13,8,5


## 01 - Graphical data visualization (optionally)

Use `data_train` only,  for better work and more illustrative graphs, replace the code names of the factor variables with the descriptions from the assignment.

* Select the categorical variables, convert them to categories, and rename coded labels according to the data description.
* Plot the discrete variables with histograms, using color to distinguish patients with and without heart disease (target 0/1).
* For continuous variables, show two boxplots by response (with vs. without heart disease) and add pairwise scatterplots of the continuous variables, coloring points by response (with/without heart disease).


## 02 - Logistic regression on aggregated tabular data

Use `data_table`.





* Define the response for a binomial logistic model and fit the **null model** (intercept only). What are the **average odds** of heart disease in the sample, and what is the **probability** of heart disease?

* Fit a model where heart disease depends **only on blood pressure**. Is blood pressure statistically significant at the 0.05 level? If yes, by how many times are the **odds** of heart disease higher for patients with blood pressure **>140** compared to those with **≤120**?

* Fit a model where heart disease depends **only on age**. Is age statistically significant at the 0.01 level? If yes, by how many times are the **odds** of heart disease higher for patients aged **>60** compared to those aged **45–60**?

* Assume the odds of heart disease increase **exponentially** with blood pressure and **exponentially** with age (equivalently, the **log-odds depend linearly** on those numeric predictors). Create corresponding **numeric continuous predictors** as the midpoints of the blood-pressure and age intervals. Fit a model where the odds depend on these numeric values **without interaction**. What is the **odds ratio** for two patients who differ by **10 years of age** but have the same blood pressure?

* Test the previous model **against the saturated model** (one parameter per age × blood-pressure cell). Does this test make sense here? Add a short comment on the result.


## 03 - Poisson regression on aggregated tabular data

Use `data_table`.


* Reshape the table into the required format and fit a **purely additive log-linear model** for the **group counts**, assuming **mutual independence** among the three grouping predictors (**age**, **blood pressure**, **disease**).

* From that model, what is the estimated **odds** of heart disease among all selected patients, and what is the estimated **probability** of heart disease?

* Fit a model that includes **all pairwise interactions** among the classification variables and compare it to the previous **no-interaction** model. Is the interaction model **significantly better**?

* Using the interaction model, what is the estimated **odds ratio** for heart disease for patients aged **>60** compared to those aged **45–60**?

* Fit the **saturated model** (one parameter per age × blood-pressure × disease cell, i.e. including the **three-way interaction**) and print the **parameter estimates**. Is this model significantly better than the model with pairwise interactions?

* Based on the saturated model, is the relationship between **blood pressure** and **heart disease** the **same across all age groups**, or does it differ?

* In which **age category** is the **largest difference** in heart disease between people with **blood pressure ≤120** and those with **blood pressure >140**?


## 04 - Logistic regression - statistical approach

Use `data_train`.

* Print a **contingency table** for `sex` and `disease`. From that table, **by hand**, compute the **empirical odds ratio** for heart disease (men vs. women) and the **probability** of disease for women and for men. Compare these to a **logistic regression** with `sex` as the only predictor and `disease` as the response. For the odds ratio, also report a **95% confidence interval**, and comment on whether women have **significantly lower odds** of heart disease.

* Print a **contingency table** for `chest_pain_type` and `disease`. From that table, **by hand**, compute the **empirical odds ratio** for heart disease comparing **type 0 (asymptomatic)** vs. **all other types**, and compute the **probability** of disease for each type. Compare these to a **logistic regression** with `chest_pain_type` as the only predictor and `disease` as the response. For the odds ratio, also report a **95% confidence interval**, and comment on whether patients with **asymptomatic** chest pain have **significantly lower odds** of heart disease than the other types.

* Fit a model using **all available variables** (both categorical and numeric). Use **deviance tests** to **stepwise reduce** the model. Compare the final model to the model you would get using **automatic stepwise selection** (e.g., `step()`).

* For your selected model, compute the **odds** of heart disease for **men vs. women**, including **95% confidence intervals**. Do the same for **asymptomatic chest pain** vs. **other types**. How did these results change compared to the simple models, and how would you **explain** the change?

* Using your model, compute **predicted probabilities** of heart disease for the **test data** and, for the predictor `blood_pressure`, **plot confidence bands** for the predicted probabilities.

* Based on the **training data**, choose a suitable **threshold** for classifying **disease vs. no disease**. On the **test data**, compute **Accuracy** and draw the **ROC curve**.




## 05 - Logistic regression - machine learning approach

Use  `data_train` and `data_test`.

* Build a **pipeline** on the **training data** for logistic regression with **elastic-net regularization** that includes:
  * Variable preparation: transformations, one-hot encoding, normalization, etc.
  * Hyper-parameter search for the “optimal” regularization settings.
  * **k-fold cross-validation**.

* Using the pipeline/workflow, choose the hyper-parameter value. If the goal is to **detect patients with heart disease**, which statistic should we focus on to avoid **sending a sick patient home as healthy** (i.e., minimize this error)?

* Compute and compare common **binary-classification metrics** on the **training** and **test** sets. Plot the **ROC curve** and compute the **AUC** for both the training and test data. What can we say about the model from **Section 05** compared to the model from **Section 04**?
